# Running hyperparameter optimization on Chemprop model using Optuna

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemprop/chemprop/blob/main/examples/hpopting_optuna.ipynb)

In [14]:
# Install chemprop from GitHub if running in Google Colab
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install ".[hpopt]"
        %cd examples

## Import packages

In [15]:
from pathlib import Path

import pandas as pd
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

import optuna

from chemprop import data, featurizers, models, nn

In [3]:
chemprop_dir = Path.cwd().parent
input_path = chemprop_dir / "tests" / "data" / "regression" / "mol" / "mol.csv" # path to your data .csv file
num_workers = 0 # number of workers for dataloader. 0 means using main process for data loading
smiles_column = 'smiles' # name of the column containing SMILES strings
target_columns = ['lipo'] # list of names of the columns containing targets

hpopt_save_dir = Path.cwd() / "hpopt" # directory to save hyperopt results
hpopt_save_dir.mkdir(exist_ok=True)

In [4]:
hpopt_save_dir

WindowsPath('C:/Users/leonz/chemprop/examples/hpopt')

## Load data

In [5]:
df_input = pd.read_csv(input_path)
df_input

,smiles,lipo
0,Cn1c(CN2CCN(CC2)c3ccc(Cl)cc3)nc4ccccc14,3.54
1,COc1cc(OC)c(cc1NC(=O)CSCC(=O)O)S(=O)(=O)N2C(C)...,-1.18
2,COC(=O)[C@@H](N1CCc2sccc2C1)c3ccccc3Cl,3.69
3,OC[C@H](O)CN1C(=O)C(Cc2ccccc12)NC(=O)c3cc4cc(C...,3.37
4,Cc1cccc(C[C@H](NC(=O)c2cc(nn2C)C(C)(C)C)C(=O)N...,3.10
...,...,...
95,CC(C)N(CCCNC(=O)Nc1ccc(cc1)C(C)(C)C)C[C@H]2O[C...,2.20
96,CCN(CC)CCCCNc1ncc2CN(C(=O)N(Cc3cccc(NC(=O)C=C)...,2.04
97,CCSc1c(Cc2ccccc2C(F)(F)F)sc3N(CC(C)C)C(=O)N(C)...,4.49
98,COc1ccc(Cc2c(N)n[nH]c2N)cc1,0.20


In [6]:
smis = df_input.loc[:, smiles_column].values
ys = df_input.loc[:, target_columns].values

## Make data points, splits, and datasets

In [7]:
all_data = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smis, ys)]

In [8]:
mols = [d.mol for d in all_data]  # RDkit Mol objects are use for structure based splits
train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)


In [9]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_dset = data.MoleculeDataset(train_data[0], featurizer)
scaler = train_dset.normalize_targets()

val_dset = data.MoleculeDataset(val_data[0], featurizer)
val_dset.normalize_targets(scaler)

test_dset = data.MoleculeDataset(test_data[0], featurizer)


# Define objective function to train the model

In [16]:
def objective(trial,build_config, train_dset, val_dset, num_workers, scaler):
    """
    This functino builds the backbone fot Optuna Hyperparameter optimization.
    """
    # Get dataloaders

    train_loader = data.build_dataloader(train_dset, num_workers=num_workers, shuffle=True)
    val_loader = data.build_dataloader(val_dset, num_workers=num_workers, shuffle=False)

    # Model

    config_dict = build_config (trial)
    mp = nn.BondMessagePassing(d_h=config_dict["mp_hidden_dim"],
                               depth=config_dict["mp_depth"])
    agg = nn.MeanAggregation()
    output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)
    ffn = nn.RegressionFFN(output_transform=output_transform,
                           hidden_dim=config_dict["ffn_hidden_dim"],
                           input_dim=config_dict["mp_hidden_dim"],
                           n_layers=config_dict["ffn_layers"],
                           )
    batch_norm = True
    metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]
    model = models.MPNN(mp, agg, ffn, batch_norm, metric_list)

    # Trainer
    checkpointing = ModelCheckpoint(
        # To get  the best val_loss from each model
        monitor="val_loss",
        mode="min",
    )
    trainer = pl.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs = 2,
        callbacks=[checkpointing],
    )
    trainer.fit(model, train_loader, val_loader)
    score = checkpointing.best_model_score
    val_loss = float("inf") if score is None else score.item()
    return val_loss


## Define parameter search space

In [17]:
def build_config (trial):
    config_dict = {
        "mp_hidden_dim": trial.suggest_int("mp_hidden_dim", 300, 2400, log=True),
        "mp_depth": trial.suggest_int("mp_depth", 2, 6, log=True),
        "ffn_hidden_dim": trial.suggest_int("ffn_hidden_dim", 300, 2400, log=True),
        "ffn_layers": trial.suggest_int("ffn_layers", 1, 3, log=True),
    }
    return config_dict

## Hyperparameters optimization

In [18]:
study = optuna.create_study (direction = "minimize")
study.optimize (lambda trial:objective (trial, build_config, train_dset, val_dset, num_workers, scaler),
                n_trials = 2,
                )


[I 2026-03-29 19:45:29,484] A new study created in memory with name: no-name-d8c5a126-56ef-4912-b8d9-1b61e5d80d90
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
C:\Users\leonz\anaconda3\envs\chemprop_env\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\leonz\anaconda3\envs\chemprop_env\Lib\site-packages\lightning\pytorch\trainer\connectors\dat

┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │ 10.4 M │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │  4.5 K │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │  1.6 M │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 12.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 12.0 M                                                                                               
Total estimated model params size (MB): 48                                                                         
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

C:\Users\leonz\anaconda3\envs\chemprop_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py
:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

...

┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  1.8 M │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │  1.8 K │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │  376 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.2 M                                                                                                
Total estimated model params size (MB): 8                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=2` reached.


[I 2026-03-29 19:45:34,210] Trial 1 finished with value: 0.8670221567153931 and parameters: {'mp_hidden_dim': 921, 'mp_depth': 2, 'ffn_hidden_dim': 306, 'ffn_layers': 2}. Best is trial 0 with value: 0.8507896661758423.


## Hyperparameter optimization results

In [27]:
study.trials

[FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.8507896661758423], datetime_start=datetime.datetime(2026, 3, 29, 19, 45, 29, 485514), datetime_complete=datetime.datetime(2026, 3, 29, 19, 45, 33, 573021), params={'mp_hidden_dim': 2243, 'mp_depth': 2, 'ffn_hidden_dim': 568, 'ffn_layers': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'mp_hidden_dim': IntDistribution(high=2400, log=True, low=300, step=1), 'mp_depth': IntDistribution(high=6, log=True, low=2, step=1), 'ffn_hidden_dim': IntDistribution(high=2400, log=True, low=300, step=1), 'ffn_layers': IntDistribution(high=3, log=True, low=1, step=1)}, trial_id=0, value=None),
 FrozenTrial(number=1, state=<TrialState.COMPLETE: 1>, values=[0.8670221567153931], datetime_start=datetime.datetime(2026, 3, 29, 19, 45, 33, 574020), datetime_complete=datetime.datetime(2026, 3, 29, 19, 45, 34, 210331), params={'mp_hidden_dim': 921, 'mp_depth': 2, 'ffn_hidden_dim': 306, 'ffn_layers': 2}, user_attrs={}, sy

In [23]:
# Best configuration
study.best_params

{'mp_hidden_dim': 2243, 'mp_depth': 2, 'ffn_hidden_dim': 568, 'ffn_layers': 2}


In [21]:
results_df = study.trials_dataframe()
results_df

,number,value,datetime_start,datetime_complete,duration,params_ffn_hidden_dim,params_ffn_layers,params_mp_depth,params_mp_hidden_dim,state
0,0,0.850790,2026-03-29 19:45:29.485514,2026-03-29 19:45:33.573021,0 days 00:00:04.087507,568,2,2,2243,COMPLETE
1,1,0.867022,2026-03-29 19:45:33.574020,2026-03-29 19:45:34.210331,0 days 00:00:00.636311,306,2,2,921,COMPLETE


In [24]:
# Save the result table
results_df.to_csv (hpopt_save_dir / "hpopt_optuna_results.tsv", sep="\t", index=False)